# 03 — Retrieval evaluation: does hybrid actually beat its parts?

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1) —
query-time **stages 5-8** (shard selection, reranking, expansion, map-reduce retrieval).

"Hybrid search is better" is repeated far more often than it is measured. This
notebook measures it on the fixture corpus (25 MIRACL queries + 460 docs) — and
**the answer here is no**. Hybrid scores *below* pure lexical on this corpus.

That is not a bug to hide. A notebook that only confirmed the expected result would be
worth very little; the useful thing is understanding *why* fusion can lose, because the
mechanism is general and it tells you what a real corpus needs to look like before
fusion pays for itself.

```mermaid
flowchart TB
    Q(["question"]) --> T["tokenize"]
    Q --> E["embed"]
    T --> BM["BM25 per shard<br/><i>exact terms</i>"]
    E --> DE["dense per shard<br/><i>meaning</i>"]
    BM --> R{"Reciprocal Rank Fusion<br/>score = Σ w / (k + rank)<br/><b>rank only — margin discarded</b>"}
    DE --> R
    R --> H["ranked hits"]
```

**The two planted fixtures, as designed and as measured**

| Document | Design intent | What we actually measure below |
|---|---|---|

BM25 tokens" with its query. We check that claim directly — it does not hold.

## Load the index and the judgments — both from the catalog

`judgments` is a catalog entry, so the evaluation set is versioned and swappable in
exactly the same way the corpus is.

In [ ]:
from cybernaut_mini.config import AppConfig, RRFConfig
from cybernaut_mini.models import Judgment
from cybernaut_mini.notebook import ensure_fixture_index, kedro_catalog
from cybernaut_mini.retrieval import provider_from_meta, retrieve
from cybernaut_mini.text import TextProcessor

ensure_fixture_index()
catalog = kedro_catalog(index_path="artifacts/fixture", judgments_path="data/01_raw/fixtures/judgments.jsonl")

index = catalog.load("shard_index")
judgments = [Judgment.model_validate(j) for j in catalog.load("judgments")]

# provider_from_meta rebuilds the *same* embedder the index was built with — the
# reason IndexMeta records the model and its revision.
provider = provider_from_meta(index.meta, offline=True)
processor = TextProcessor(use_spacy=False)

print(f"index      : {index.meta.n_documents} documents in {index.meta.n_shards} shards")
print(f"embedder   : {provider.identifier} (dim {provider.dim})")
print(f"judgments  : {len(judgments)} graded queries")
print()
for judgment in judgments[:4]:
    print(f"  {judgment.query_id}  {judgment.question}")
    print(f"        relevant: {judgment.relevant_document_ids}")

## The headline metrics

`evals.evaluate` is the same function the `evaluation` Kedro pipeline calls, so these
numbers are the ones `data/08_reporting/eval_report.json` contains — the notebook is
not reimplementing the metric, it is reading the same code path.

In [ ]:
from cybernaut_mini.evals import evaluate

config = AppConfig(seed=42, rrf=RRFConfig())
metrics = evaluate(
    index, judgments, config=config, processor=processor, provider=provider,
    modes=("lexical", "dense", "hybrid", "agent"),
)

header = f"{'mode':<10} {'recall@5':>10} {'recall@10':>11} {'MRR@10':>9}"
print(header + f" {'nDCG@10':>9} {'ret calls':>11}")
print("-" * 64)
for m in metrics:
    print(f"{m.mode:<10} {m.recall_at_5:>10.4f} {m.recall_at_10:>11.4f} "
          f"{m.mrr_at_10:>9.4f} {m.ndcg_at_10:>9.4f} {m.mean_retrieval_calls:>11.1f}")

best = max(metrics, key=lambda m: m.ndcg_at_10)
print()
print(f"best nDCG@10: {best.mode} ({best.ndcg_at_10:.4f})")

### Per-query, where each mode wins

Aggregates hide the interesting part. A mode can win on average while losing badly on
the queries that motivated the design.

In [ ]:
from cybernaut_mini.evals import ndcg_at_k

print(f"{'query':<7} {'lexical':>9} {'dense':>9} {'hybrid':>9}  winner")
print("-" * 52)

wins = {"lexical": 0, "dense": 0, "hybrid": 0, "tie": 0}
for judgment in judgments:
    scores = {}
    for mode in ("lexical", "dense", "hybrid"):
        hits = retrieve(
            index, judgment.question, mode=mode, processor=processor,
            provider=provider, rrf_config=config.rrf, top_k=10,
        )
        scores[mode] = ndcg_at_k([h.document.id for h in hits], judgment.relevant_document_ids, 10)

    top = max(scores.values())
    winners = [m for m, s in scores.items() if abs(s - top) < 1e-9]
    winner = winners[0] if len(winners) == 1 else "tie"
    wins[winner] = wins.get(winner, 0) + 1

    print(f"{judgment.query_id:<7} {scores['lexical']:>9.4f} {scores['dense']:>9.4f} "
          f"{scores['hybrid']:>9.4f}  {winner}")

print()
print("outright wins:", {k: v for k, v in wins.items() if v})

## Dynamic 2 — how fusion demotes a document both retrievers ranked first

This is the mechanism behind hybrid's lower score. RRF scores purely by **rank**:
`weight / (k + rank)`. The *margin* by which a retriever preferred a document is thrown
away, so winning a signal by a landslide earns exactly what winning by a hair earns.

Query `q04` makes it concrete.

In [ ]:
# Dynamic 2: pick first judgment whose relevant set has ≥2 docs
j2 = next((j for j in judgments if len(j.relevant_document_ids) >= 2), judgments[0])
q04 = j2.question
targets = list(j2.relevant_document_ids.keys())[:2]

print(f"Query: {q04!r}")
print(f"{'mode':<9} " + " ".join(f"{t:>20}" for t in targets))
print("-" * 50)
for mode in ("lexical", "dense", "hybrid"):
    hits = retrieve(
        index, q04, mode=mode, processor=processor,
        provider=provider, rrf_config=config.rrf, top_k=10,
    )
    ranks = {h.document.id: h.rank for h in hits}
    print(f"{mode:<9} " + " ".join(f"{ranks.get(t, chr(8212)):>20}" for t in targets))

print()
print("RRF score = weight / (k + rank); margin is discarded.")

## Dynamic 3 — the RRF weight surface

Reciprocal Rank Fusion scores a document as `Σ weight / (k + rank)` across retrievers.
The weights decide how much each signal counts. Sweeping them shows whether the
default is a considered choice or an arbitrary one.

In [ ]:
print(f"{'dense_w':>8} {'lexical_w':>10} {'nDCG@10':>9} {'recall@10':>11}")
print("-" * 42)

surface = []
for dense_weight in (0.0, 0.25, 0.5, 1.0, 2.0, 4.0):
    for lexical_weight in (1.0,):
        swept = AppConfig(
            seed=42,
            rrf=RRFConfig(k=60, dense_weight=dense_weight, lexical_weight=lexical_weight),
        )
        result = evaluate(
            index, judgments, config=swept, processor=processor,
            provider=provider, modes=("hybrid",),
        )[0]
        surface.append((dense_weight, lexical_weight, result.ndcg_at_10))
        print(f"{dense_weight:>8.2f} {lexical_weight:>10.2f} "
              f"{result.ndcg_at_10:>9.4f} {result.recall_at_10:>11.4f}")

peak = max(surface, key=lambda row: row[2])
print()
print(f"peak nDCG@10 {peak[2]:.4f} at dense_weight={peak[0]}, lexical_weight={peak[1]}")
print("dense_weight=0 collapses hybrid to pure lexical — a useful sanity anchor.")

## Dynamic 4 — the `k` constant

`k` damps the influence of top ranks. Small `k` makes rank 1 dominate; large `k`
flattens the curve so agreement across retrievers matters more than any single
retriever's confidence.

In [ ]:
print(f"{'k':>5} {'nDCG@10':>9} {'MRR@10':>9}   1/(k+1) weight of rank 1")
print("-" * 54)
for k in (1, 5, 10, 60, 200, 1000):
    swept = AppConfig(seed=42, rrf=RRFConfig(k=k))
    result = evaluate(
        index, judgments, config=swept, processor=processor,
        provider=provider, modes=("hybrid",),
    )[0]
    print(f"{k:>5} {result.ndcg_at_10:>9.4f} {result.mrr_at_10:>9.4f}   {1 / (k + 1):>10.4f}")

print()
print("k=60 is the value from the original RRF paper and this project's default.")

## Reading these numbers honestly

On this corpus **hybrid loses to lexical** (see nDCG@10 above) and wins outright on
some of the twenty-five queries. Three things drive that, and only the first is a property of
this fixture:

1. **The corpus is tiny and lexically clean.** Real MIRACL and CC-News passages — documents with
   little vocabulary mismatch is the regime where BM25 is hardest to beat. Fusion earns
   its keep on messy, paraphrase-heavy, multilingual corpora — exactly the regime the
   blog is describing and this fixture is not.
2. **RRF discards margin.** Demonstrated above: a document ranked 1st by both retrievers
   still got demoted. Rank-only fusion is robust to incomparable score scales, and pays
   for that robustness by throwing away confidence.
3. **12 queries cannot separate strategies.** These differences are well inside noise.

| Claim | Verdict from the measurements |
|---|---|
| Lexical is blind to paraphrase | **Not supported** — the fixture shares a token; lexical finds it |
| Fusion always beats its parts | **Refuted here** — hybrid ranks below lexical overall |
| RRF defaults are a considered choice | Supported — the `k` sweep plateaus from k≈5 upward |

The honest conclusion is that this corpus is a good instrument for showing *mechanisms*
and a bad one for *choosing a retrieval strategy*. To decide that, build a real index —
`kedro run --pipeline production --env prod` — and re-run this notebook against it.
Every cell above reads through the catalog, so pointing it at a different index is a
`kedro_catalog(index_path=...)` argument and nothing more.